In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import librosa
import numpy as np
import os
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

In [ ]:
def extract_spectrogram(file_path):
    y, sr = librosa.load(file_path, duration=30)
    
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    
    mel_db = mel_db[:128, :128]
    
    # 🔥 SAFE NORMALIZATION
    mean = np.mean(mel_db)
    std = np.std(mel_db)
    
    if std != 0:
        mel_db = (mel_db - mean) / std
    else:
        mel_db = mel_db - mean
    
    return mel_db

In [ ]:
base_path = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"

X = []
y = []

for genre in os.listdir(base_path):
    genre_path = os.path.join(base_path, genre)
    
    for song in tqdm(os.listdir(genre_path)[:100]):   # more data
        
        file_path = os.path.join(genre_path, song, "vocals.wav")
        
        try:
            spec = extract_spectrogram(file_path)
            X.append(spec)
            y.append(genre)
        except:
            continue

X = np.array(X)
y = np.array(y)

print("Dataset:", X.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_enc = le.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y_enc, test_size=0.2, random_state=42
)

In [ ]:
X_train = torch.tensor(X_train).unsqueeze(1).float()
X_val = torch.tensor(X_val).unsqueeze(1).float()

y_train = torch.tensor(y_train).long()
y_val = torch.tensor(y_val).long()

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(32, 64, 3, padding=1),   # 🔥 NEW
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 10)
        )
    
    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

In [ ]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0005)

min_loss = float('inf')   # start with very large number

for epoch in range(30):
    
    model.train()
    
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    # update minimum loss
    if loss.item() < min_loss:
        min_loss = loss.item()

# print only final minimum loss
print(f" Minimum Loss: {min_loss}")

In [ ]:
model.eval()

with torch.no_grad():
    preds = model(X_val)
    preds = torch.argmax(preds, axis=1)

f1 = f1_score(y_val.numpy(), preds.numpy(), average="macro")

print(" CNN Macro F1:", f1)

# CNN Macro F1: 0.3948965885195995